# Layer 06 - Chatbot E2E Verification

Companion to `04_rag_search_gemma_verification.ipynb`. **`04` already covers the deterministic Cypher / Neo4j data-layer regression** (hand-curated `(weights, filters)` → exact rows match golden). This notebook tests **what `04` doesn't**: the chatbot layer itself.

Specifically:

| Section | Tests |
|---|---|
| 1 | Connectivity sanity (backend, Ollama, Neo4j, both demo logins) |
| 2 | LLM Stage-1 intent + filter extraction quality (~20 cases) |
| 3 | Tool dispatch correctness (predict / CBR / SHAP / shortlist / town overlap / school cross-ref / combined) |
| 4 | Framing & hallucination guards (negative honesty, PropertyGuru redirect, anti-invention) |
| 5 | Per-user authorization (JWT scoping → different shortlist responses for different users) |
| 6 | Latency profile per surface (smalltalk / shortlist / tool / search) |
| 7 | Combined headline summary |

Pass criteria are categorical and structural — section headers, key-field regexes, intent classification — not bit-exact row identity. That kind of identity test is `04`'s job and is meaningless once an LLM stage sits in front of the Cypher.

Total runtime: ~3-5 minutes (~33 chatbot calls vs the 100+10 in the previous version of this notebook).

Pre-conditions:
- Backend running on `:8000` (with today's framing fix, shortlist projection, school cross-reference)
- Ollama with `gemma3` available
- Neo4j with all 5 node labels populated (`Property`, `Town`, `FamousSchool`, `User`, `UserShortlistItem`)
- Demo users `bhuvesh` (has BLK 864 TAMPINES ST 83 saved → exercises shortlist + cross-ref) and `user` (has UBI saves → exercises Maha Bodhi cross-ref) both with `password=1234`

In [12]:
from __future__ import annotations

import json
import re
import sys
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

API_BASE = 'http://localhost:8000'


def login(username: str, password: str = '1234') -> str:
    r = requests.post(
        f'{API_BASE}/api/auth/login',
        json={'username': username, 'password': password},
        timeout=15,
    )
    r.raise_for_status()
    return r.json()['access_token']


JWT_BHUVESH = login('bhuvesh')
JWT_USER    = login('user')
print(f'Logged in: bhuvesh ({len(JWT_BHUVESH)} chars), user ({len(JWT_USER)} chars)')


_DECODE = lambda s: s.replace('\\\\', '\\').replace('\\n', '\n')


def post_chat(message: str, *, jwt: str | None = None, timeout_s: int = 60) -> dict:
    """POST to /api/property-search-chat, parse the SSE stream."""
    headers = {'Content-Type': 'application/json'}
    if jwt:
        headers['Authorization'] = f'Bearer {jwt}'
    t0 = time.perf_counter()
    r = requests.post(
        f'{API_BASE}/api/property-search-chat',
        headers=headers,
        json={'message': message, 'history': []},
        stream=True,
        timeout=timeout_s,
    )
    r.raise_for_status()
    text_parts: list[str] = []
    params: dict | None = None
    for raw in r.iter_lines(decode_unicode=True):
        if not raw or not raw.startswith('data:'):
            continue
        body = raw[5:].strip()
        if body == '[DONE]':
            break
        if body.startswith('[LOG]') or body.startswith('[STATUS]'):
            continue
        if body.startswith('[PARAMS]') and body.endswith('[/PARAMS]'):
            try:
                params = json.loads(body[len('[PARAMS]'):-len('[/PARAMS]')])
            except Exception:
                pass
            continue
        text_parts.append(_DECODE(body))
    return {
        'markdown': '\n'.join(text_parts).strip(),
        'params': params or {},
        'latency_ms': int((time.perf_counter() - t0) * 1000),
    }


_SECTION_HEADERS = {
    'predict':       '### Price estimate',
    'cbr':           '### Similar past sales',
    'shap':          '### SHAP drivers',
    'shortlist':     '### Your shortlist',
    'town_overlap':  '### Your saves by town',
    'school_xref':   '### From your shortlist near',
    'historical':    '### Historical comparables',
    'historical_school': '### Historical comparables near',
}


def classify_response(md: str, params: dict | None = None) -> str:
    """Infer which chatbot branch fired from the rendered markdown."""
    md_l = md.lower()
    if not md.strip():
        return 'empty'
    # Negative-result search responses look short and have no headers but
    # are NOT smalltalk — the LLM honestly said "no matches" or the chatbot's
    # fallback path fired. Detect by phrase before defaulting to smalltalk.
    NEG_PATTERNS = (
        'no matching historical sales found',
        "couldn't find matching properties",
        'couldn’t find matching properties',
        'property search is unavailable',
    )
    is_negative_search = any(p in md_l for p in NEG_PATTERNS)
    if is_negative_search:
        # Distinguish school path by Stage-1 special_query_type
        return 'search_school' if (params or {}).get('special_query_type') == 'near_famous_school' else 'search'
    if '###' not in md and len(md) < 250:
        return 'smalltalk'
    # Tool sections take precedence over generic search
    if _SECTION_HEADERS['predict'].lower() in md_l:
        return 'predict'
    if _SECTION_HEADERS['cbr'].lower() in md_l:
        return 'cbr'
    if _SECTION_HEADERS['shap'].lower() in md_l:
        return 'shap'
    if _SECTION_HEADERS['shortlist'].lower() in md_l:
        return 'shortlist'
    if (params or {}).get('special_query_type') == 'near_famous_school':
        return 'search_school'
    if _SECTION_HEADERS['historical'].lower() in md_l:
        return 'search'
    return 'unknown'


_ROW_ADDR_RE = re.compile(r'\*\*\d+\.\s+([A-Z0-9][A-Z0-9 .\'-]+?)\*\*')
_NARRATIVE_ADDR_RE = re.compile(r'\b(\d{1,4}[A-Z]?)\s+([A-Z][A-Z0-9 .\'\-]+?\s(?:ST|AVE|RD|DR|CRES|CTRL|CL|PL|TER|HTS|GDNS|WALK|LANE|LINK|RISE|VIEW|FIELD|EAST|NTH|STH|WAY)\b\s*\d*)', re.IGNORECASE)


def addresses_from_rows(md: str) -> list[str]:
    """Extract address keys from the `### Historical comparables` row table only."""
    if not md:
        return []
    idx = md.find('### Historical comparables')
    section = md[idx:] if idx >= 0 else md
    return [m.group(1).strip() for m in _ROW_ADDR_RE.finditer(section)]


def addresses_from_narrative(md: str) -> list[str]:
    """Loosely parse addresses mentioned in the LLM narrative paragraph.

    The narrative is the text AFTER the `### Historical comparables` block.
    Used by the advisory anti-hallucination check.
    """
    if not md:
        return []
    # Narrative is whatever follows the row table — find the last `\n\n` after
    # the comparables header, then everything after that.
    idx = md.find('### Historical comparables')
    if idx < 0:
        return []
    # End of the row block: find first non-bullet paragraph after the header
    rest = md[idx:]
    paras = rest.split('\n\n')
    # Skip the header + the row block (lines starting with `- `)
    narrative_paras = [p for p in paras[2:] if p.strip() and not p.lstrip().startswith('- ')]
    text = ' '.join(narrative_paras)
    matches = _NARRATIVE_ADDR_RE.findall(text)
    return [f'{blk} {street}'.strip() for blk, street in matches]


print('Setup helpers ready.')

Logged in: bhuvesh (127 chars), user (123 chars)
Setup helpers ready.


## 1. Connectivity sanity

Surface infrastructure problems before the rest of the notebook gives confusing errors.

In [13]:
checks = []

# Backend health
try:
    h = requests.get(f'{API_BASE}/health', timeout=5)
    checks.append({'check': 'backend_health', 'pass': h.status_code == 200, 'detail': f'HTTP {h.status_code} in {int(h.elapsed.total_seconds()*1000)}ms'})
except Exception as e:
    checks.append({'check': 'backend_health', 'pass': False, 'detail': f'{type(e).__name__}: {e}'})

# Ollama tags
try:
    o = requests.get('http://127.0.0.1:11434/api/tags', timeout=5)
    has_gemma = any('gemma3' in (m.get('name') or '') for m in (o.json().get('models') or []))
    checks.append({'check': 'ollama_gemma3', 'pass': o.status_code == 200 and has_gemma, 'detail': f'HTTP {o.status_code}, gemma3 listed: {has_gemma}'})
except Exception as e:
    checks.append({'check': 'ollama_gemma3', 'pass': False, 'detail': f'{type(e).__name__}: {e}'})

# Neo4j node counts (matches 04's snapshot expectations)
try:
    from yc_property_search import Neo4jPropertySearch
    with Neo4jPropertySearch() as neo:
        live = neo.node_counts()
    expected = {'Property': 9710, 'Town': 26, 'FamousSchool': 16}
    counts_ok = all(live.get(k) == v for k, v in expected.items())
    checks.append({'check': 'neo4j_node_counts', 'pass': counts_ok, 'detail': f'{live} vs expected {expected}'})
except Exception as e:
    checks.append({'check': 'neo4j_node_counts', 'pass': False, 'detail': f'{type(e).__name__}: {e}'})

# Both demo logins
checks.append({'check': 'login_bhuvesh', 'pass': bool(JWT_BHUVESH), 'detail': f'token len {len(JWT_BHUVESH)}'})
checks.append({'check': 'login_user',    'pass': bool(JWT_USER),    'detail': f'token len {len(JWT_USER)}'})

connectivity = pd.DataFrame(checks)
display(connectivity)
if not connectivity['pass'].all():
    print('⚠️  Connectivity failures — fix before running the rest of this notebook.')
else:
    print('✓ All connectivity checks passed.')

,check,pass,detail
0,backend_health,True,HTTP 200 in 3ms
1,ollama_gemma3,True,"HTTP 200, gemma3 listed: True"
2,neo4j_node_counts,True,"{'Property': 9710, 'Town': 26, 'FamousSchool':..."
3,login_bhuvesh,True,token len 127
4,login_user,True,token len 123


✓ All connectivity checks passed.


## 2. LLM Stage-1 extraction quality

20 hand-curated cases. Each asserts that the chatbot's branch dispatch (intent classification) and the categorical filters (town, flat_type, school_name) are extracted correctly. We do NOT assert exact weight magnitudes — those drift, and that's expected for an LLM. We DO assert `max_resale_price` is within ±25% of the user's number, which catches order-of-magnitude misreads like dropping a zero.

If `intent_match` ≥ 18/20 (90%) and `filter_match` ≥ 18/20, the chatbot's NL understanding is healthy.

In [14]:
STAGE1_CASES = [
    # smalltalk
    {'q': 'Hi',                                       'intent': 'smalltalk'},
    {'q': 'thanks',                                   'intent': 'smalltalk'},
    # shortlist
    {'q': 'my shortlist',                             'intent': 'shortlist'},
    {'q': 'what is in my wishlist',                   'intent': 'shortlist'},
    {'q': 'my saves by town vs historical median',    'intent': 'shortlist'},
    # tools (require flat extraction)
    {'q': 'Predict price for BLK 864 Tampines Street 83, 4 room, 122 sqm, floor 20, lease commenced 1987',
     'intent': 'predict'},
    {'q': 'Show me 5 similar past sales for BLK 864 Tampines Street 83, 4 room, 122 sqm',
     'intent': 'cbr'},
    {'q': 'Why is BLK 864 Tampines Street 83 priced that way? 4 room, 122 sqm, floor 20',
     'intent': 'shap'},
    # search — town + flat_type extraction
    {'q': 'Find a 4-room flat in Bishan under $900k near MRT',
     'intent': 'search', 'town': 'BISHAN', 'flat_type': '4 ROOM',
     'max_resale_price': 900000},
    {'q': 'Show me 5-room flats in Punggol with at least 100 sqm',
     'intent': 'search', 'town': 'PUNGGOL', 'flat_type': '5 ROOM'},
    {'q': 'Find a 3 room flat in Hougang under $500k',
     'intent': 'search', 'town': 'HOUGANG', 'flat_type': '3 ROOM',
     'max_resale_price': 500000},
    {'q': 'Recommend a 4-room flat with the best access to top primary schools',
     'intent': 'search', 'flat_type': '4 ROOM'},
    {'q': 'I want a quiet 5 room flat in Tampines with at least 120 sqm and at least 70 years lease left',
     'intent': 'search', 'town': 'TAMPINES', 'flat_type': '5 ROOM'},
    {'q': 'Show me a 4 room flat in Bishan with at least 90 sqm, within 500m of MRT, under $800k',
     'intent': 'search', 'town': 'BISHAN', 'flat_type': '4 ROOM',
     'max_resale_price': 800000},
    # search — special school path
    {'q': 'Show me properties near POI CHING SCHOOL',
     'intent': 'search_school', 'school_name': 'POI CHING SCHOOL'},
    {'q': 'Show me flats near NANYANG PRIMARY SCHOOL',
     'intent': 'search_school', 'school_name': 'NANYANG PRIMARY SCHOOL'},
    {'q': 'Show me flats near MAHA BODHI SCHOOL',
     'intent': 'search_school', 'school_name': 'MAHA BODHI SCHOOL'},
    {'q': 'Show me flats near ROSYTH SCHOOL',
     'intent': 'search_school', 'school_name': 'ROSYTH SCHOOL'},
    # negative — should still classify as search, just with no rows
    {'q': 'Find a 3 room flat in Bishan under $100k',
     'intent': 'search', 'town': 'BISHAN', 'flat_type': '3 ROOM',
     'max_resale_price': 100000},
    # "where can I buy" — should classify as search but NOT recommend (LLM should redirect)
    {'q': 'Where can I buy a 4-room in Bedok right now?',
     'intent': 'search', 'town': 'BEDOK', 'flat_type': '4 ROOM'},
]


def evaluate_stage1(case: dict) -> dict:
    try:
        res = post_chat(case['q'], jwt=JWT_BHUVESH)
    except Exception as e:
        return {**{'q': case['q'][:60], 'expected_intent': case['intent']},
                'pass': False, 'error': f'{type(e).__name__}: {e}', 'latency_ms': None}
    md, params = res['markdown'], res['params']
    actual_intent = classify_response(md, params)
    intent_match = actual_intent == case['intent']
    filters = (params.get('filters') or {}) if isinstance(params, dict) else {}

    town_ok = (case.get('town') is None) or (str(filters.get('town', '')).upper() == case['town'])
    flat_type_ok = (case.get('flat_type') is None) or (str(filters.get('flat_type', '')).upper() == case['flat_type'])
    school_ok = (case.get('school_name') is None) or (str(filters.get('school_name', '')).upper() == case['school_name'])
    if case.get('max_resale_price') is None:
        budget_ok = True
    else:
        actual_budget = filters.get('max_resale_price')
        if actual_budget is None:
            budget_ok = False
        else:
            tol = 0.25 * case['max_resale_price']
            budget_ok = abs(float(actual_budget) - case['max_resale_price']) <= tol

    filter_match = town_ok and flat_type_ok and school_ok and budget_ok
    return {
        'q': case['q'][:55] + ('...' if len(case['q']) > 55 else ''),
        'expected_intent': case['intent'],
        'actual_intent': actual_intent,
        'intent_match': intent_match,
        'town_ok': town_ok,
        'flat_type_ok': flat_type_ok,
        'school_ok': school_ok,
        'budget_ok': budget_ok,
        'filter_match': filter_match,
        'pass': intent_match and filter_match,
        'latency_ms': res['latency_ms'],
        'markdown': res['markdown'][:600] + ('...' if len(res['markdown']) > 600 else ''),
    }

In [15]:
stage1 = pd.DataFrame(evaluate_stage1(c) for c in STAGE1_CASES)
display(stage1)
print(f'Intent matches : {int(stage1["intent_match"].sum())}/{len(stage1)}')
print(f'Filter matches : {int(stage1["filter_match"].sum())}/{len(stage1)}')
print(f'Combined pass  : {int(stage1["pass"].sum())}/{len(stage1)}')

,q,expected_intent,actual_intent,intent_match,town_ok,flat_type_ok,school_ok,budget_ok,filter_match,pass,latency_ms,markdown
0,Hi,smalltalk,smalltalk,True,True,True,True,True,True,True,6,"Hi — I can help you search for HDB flats, esti..."
1,thanks,smalltalk,smalltalk,True,True,True,True,True,True,True,3,"You're welcome — want to search for flats, pre..."
2,my shortlist,shortlist,shortlist,True,True,True,True,True,True,True,57,### Your shortlist (graph view — vs historical...
3,what is in my wishlist,shortlist,shortlist,True,True,True,True,True,True,True,115,### Your shortlist (graph view — vs historical...
4,my saves by town vs historical median,shortlist,shortlist,True,True,True,True,True,True,True,88,### Your shortlist (graph view — vs historical...
5,"Predict price for BLK 864 Tampines Street 83, ...",predict,predict,True,True,True,True,True,True,True,88,### Price estimate\n\nHybrid model predicted S...
6,Show me 5 similar past sales for BLK 864 Tampi...,cbr,cbr,True,True,True,True,True,True,True,54,### Similar past sales\n\n- TAMPINES BLK 881 T...
7,Why is BLK 864 Tampines Street 83 priced that ...,shap,shap,True,True,True,True,True,True,True,92,### SHAP drivers\n\ntype=local predicted_hybri...
8,Find a 4-room flat in Bishan under $900k near MRT,search,search,True,True,True,True,True,True,True,4395,### Historical comparables matching your crite...
9,Show me 5-room flats in Punggol with at least ...,search,search,True,False,True,True,True,False,False,2061,I couldn’t find matching properties in the dat...


Intent matches : 20/20
Filter matches : 19/20
Combined pass  : 19/20


In [16]:
# Show full markdown response for any Stage-1 failure — primary diagnostic surface.
stage1_failures = stage1.loc[~stage1['pass']]
if len(stage1_failures):
    print(f'{len(stage1_failures)} Stage-1 failure(s):\n')
    for _, row in stage1_failures.iterrows():
        print('─' * 80)
        print(f"Q          : {row['q']}")
        print(f"Expected   : {row['expected_intent']}")
        print(f"Actual     : {row['actual_intent']}")
        print(f"Filter ok  : town={row['town_ok']} flat_type={row['flat_type_ok']} school={row['school_ok']} budget={row['budget_ok']}")
        print(f"Latency    : {row['latency_ms']}ms")
        print(f"Markdown   :\n{row['markdown']}")
else:
    print('No Stage-1 failures.')

1 Stage-1 failure(s):

────────────────────────────────────────────────────────────────────────────────
Q          : Show me 5-room flats in Punggol with at least 100 sqm
Expected   : search
Actual     : search
Filter ok  : town=False flat_type=True school=True budget=True
Latency    : 2061ms
Markdown   :
I couldn’t find matching properties in the database for that query.


## 3. Tool dispatch correctness

Each tool has a fixture asserting (a) the right section header appears and (b) a key field is present in the response. If any of these fail, a tool is broken end-to-end.

In [17]:
TOOL_CASES = [
    {
        'name': 'predict',
        'q': 'Predict the price for BLK 864 Tampines Street 83, 4 room, 122 sqm, floor 20, lease commenced 1987',
        'jwt': JWT_BHUVESH,
        'expects_section': '### Price estimate',
        'key_field_re': r'(?:SGD|\$)\s?\d+,\d{3}',
    },
    {
        'name': 'cbr',
        'q': 'Show me 5 similar past sales for BLK 864 Tampines Street 83, 4 room, 122 sqm',
        'jwt': JWT_BHUVESH,
        'expects_section': '### Similar past sales',
        'min_row_count': 3,
    },
    {
        'name': 'shap',
        'q': 'Why is BLK 864 Tampines Street 83 priced that way? 4 room, 122 sqm, floor 20',
        'jwt': JWT_BHUVESH,
        'expects_section': '### SHAP drivers',
        'must_contain_all': ['floor_area_sqm', 'transaction_year'],
    },
    {
        'name': 'shortlist_graph',
        'q': 'my shortlist',
        'jwt': JWT_BHUVESH,
        'expects_section': '### Your shortlist (graph view',
        'must_contain_all': ['asking ', 'AI ', 'hist '],
    },
    {
        'name': 'town_overlap',
        'q': 'my saves by town vs historical median',
        'jwt': JWT_BHUVESH,
        'expects_section': '### Your saves by town vs historical median',
        'must_contain_all': ['Saves', 'Historical median'],
    },
    {
        'name': 'school_xref',
        'q': 'Show me properties near POI CHING SCHOOL',
        'jwt': JWT_BHUVESH,
        'expects_section': '### From your shortlist near POI CHING SCHOOL',
        'expects_secondary': '### Historical comparables near POI CHING SCHOOL',
        'must_contain_all': ['864 TAMPINES ST 83'],
    },
    {
        'name': 'predict_from_shortlist',
        'q': 'Predict the price for the latest flat in my shortlist',
        'jwt': JWT_BHUVESH,
        'expects_section': '### Price estimate',
        'key_field_re': r'(?:SGD|\$)\s?\d+,\d{3}',
    },
]


def evaluate_tool(case: dict) -> dict:
    try:
        res = post_chat(case['q'], jwt=case['jwt'])
    except Exception as e:
        return {'name': case['name'], 'pass': False, 'error': f'{type(e).__name__}: {e}', 'latency_ms': None}
    md = res['markdown']
    section_ok = case['expects_section'].lower() in md.lower()
    secondary_ok = (case.get('expects_secondary', '').lower() in md.lower()) if case.get('expects_secondary') else True
    key_field_ok = bool(re.search(case['key_field_re'], md)) if case.get('key_field_re') else True
    must_all_ok = all(s.lower() in md.lower() for s in case.get('must_contain_all', []))
    row_count_ok = True
    if case.get('min_row_count'):
        idx = md.lower().find(case['expects_section'].lower())
        section = md[idx:] if idx >= 0 else ''
        row_lines = [ln for ln in section.split('\n') if ln.lstrip().startswith('- ')]
        row_count_ok = len(row_lines) >= case['min_row_count']
    return {
        'name': case['name'],
        'q': case['q'][:55] + ('...' if len(case['q']) > 55 else ''),
        'section_ok': section_ok,
        'secondary_ok': secondary_ok,
        'key_field_ok': key_field_ok,
        'must_all_ok': must_all_ok,
        'row_count_ok': row_count_ok,
        'pass': section_ok and secondary_ok and key_field_ok and must_all_ok and row_count_ok,
        'latency_ms': res['latency_ms'],
        'markdown': res['markdown'][:600] + ('...' if len(res['markdown']) > 600 else ''),
    }

In [18]:
tools = pd.DataFrame(evaluate_tool(c) for c in TOOL_CASES)
display(tools)
print(f'Tool dispatch: {int(tools["pass"].sum())}/{len(tools)}')

,name,q,section_ok,secondary_ok,key_field_ok,must_all_ok,row_count_ok,pass,latency_ms,markdown
0,predict,Predict the price for BLK 864 Tampines Street ...,True,True,True,True,True,True,96,### Price estimate\n\nHybrid model predicted S...
1,cbr,Show me 5 similar past sales for BLK 864 Tampi...,True,True,True,True,True,True,54,### Similar past sales\n\n- TAMPINES BLK 881 T...
2,shap,Why is BLK 864 Tampines Street 83 priced that ...,True,True,True,True,True,True,107,### SHAP drivers\n\ntype=local predicted_hybri...
3,shortlist_graph,my shortlist,True,True,True,True,True,True,112,### Your shortlist (graph view — vs historical...
4,town_overlap,my saves by town vs historical median,True,True,True,True,True,True,86,### Your shortlist (graph view — vs historical...
5,school_xref,Show me properties near POI CHING SCHOOL,True,True,True,True,True,True,5207,### From your shortlist near POI CHING SCHOOL\...
6,predict_from_shortlist,Predict the price for the latest flat in my sh...,True,True,True,True,True,True,150,### Your shortlist (graph view — vs historical...


Tool dispatch: 7/7


In [19]:
# Show markdown for any tools failure.
_failures = tools.loc[~tools['pass']]
if len(_failures):
    print(f'{len(_failures)} tools failure(s):\n')
    for _, row in _failures.iterrows():
        print('─' * 80)
        print(f"name       : {row.get('name', '')}")
        print(f"q          : {row.get('q', '')}")
        print(f"section_ok : {row.get('section_ok', '')}")
        print(f"key_field_ok : {row.get('key_field_ok', '')}")
        print(f"must_all_ok : {row.get('must_all_ok', '')}")
        print(f"latency_ms : {row.get('latency_ms', '')}")
        print(f"markdown   :\n{row.get('markdown', '(no markdown captured)')}")
else:
    print('No tools failures.')

No tools failures.


## 4. Framing & hallucination guards

Tests the framing fixes from earlier today.

- F1: negative-result query → chatbot says 'no matching historical sales' instead of inventing addresses.
- F2: 'where can I buy' query → LLM redirects to PropertyGuru / 99.co (per the Stage-3 system prompt).
- F3: positive search query → response has `### Historical comparables` AND `sold ` framing visible.
- F4 (advisory): for any positive-result query, parse addresses mentioned in the LLM narrative paragraph; warn (don't fail) if any aren't in the row table — surfaces residual LLM drift.

In [20]:
FRAMING_CASES = [
    {'name': 'F1_negative_honesty',
     'q': 'Find a 3 room flat in Bishan under $100k',
     'must_contain_any_lc': ['no matching historical sales found', "couldn't find matching properties", 'couldn\u2019t find matching properties'],
     'must_not_contain_lc': ['**1.']},
    {'name': 'F2_where_can_i_buy_redirect',
     'q': 'Where can I buy a 4-room in Bedok right now?',
     'must_contain_any_lc': ['propertyguru', '99.co', 'current listing']},
    {'name': 'F3_historical_framing_visible',
     'q': 'Find a 4-room flat in Bishan under $900k near MRT',
     'must_contain_any_lc': ['### historical comparables'],
     'must_contain_all_lc': ['sold ']},
]


def evaluate_framing(case: dict) -> dict:
    try:
        res = post_chat(case['q'], jwt=JWT_BHUVESH)
    except Exception as e:
        return {'name': case['name'], 'pass': False, 'error': f'{type(e).__name__}: {e}', 'latency_ms': None}
    md_lc = res['markdown'].lower()
    any_ok = any(s in md_lc for s in case.get('must_contain_any_lc', [])) if case.get('must_contain_any_lc') else True
    all_ok = all(s in md_lc for s in case.get('must_contain_all_lc', [])) if case.get('must_contain_all_lc') else True
    not_ok = all(s not in md_lc for s in case.get('must_not_contain_lc', [])) if case.get('must_not_contain_lc') else True
    return {
        'name': case['name'],
        'q': case['q'][:55] + ('...' if len(case['q']) > 55 else ''),
        'any_ok': any_ok,
        'all_ok': all_ok,
        'not_ok': not_ok,
        'pass': any_ok and all_ok and not_ok,
        'latency_ms': res['latency_ms'],
        'markdown': res['markdown'][:600] + ('...' if len(res['markdown']) > 600 else ''),
    }


framing = pd.DataFrame(evaluate_framing(c) for c in FRAMING_CASES)
display(framing)
print(f'Framing: {int(framing["pass"].sum())}/{len(framing)}')

,name,q,any_ok,all_ok,not_ok,pass,latency_ms,markdown
0,F1_negative_honesty,Find a 3 room flat in Bishan under $100k,True,True,True,True,1995,I couldn’t find matching properties in the dat...
1,F2_where_can_i_buy_redirect,Where can I buy a 4-room in Bedok right now?,True,True,True,True,3466,### Historical comparables matching your crite...
2,F3_historical_framing_visible,Find a 4-room flat in Bishan under $900k near MRT,True,True,True,True,5054,### Historical comparables matching your crite...


Framing: 3/3


In [21]:
# Show markdown for any framing failure.
_failures = framing.loc[~framing['pass']]
if len(_failures):
    print(f'{len(_failures)} framing failure(s):\n')
    for _, row in _failures.iterrows():
        print('─' * 80)
        print(f"name       : {row.get('name', '')}")
        print(f"q          : {row.get('q', '')}")
        print(f"any_ok     : {row.get('any_ok', '')}")
        print(f"all_ok     : {row.get('all_ok', '')}")
        print(f"not_ok     : {row.get('not_ok', '')}")
        print(f"latency_ms : {row.get('latency_ms', '')}")
        print(f"markdown   :\n{row.get('markdown', '(no markdown captured)')}")
else:
    print('No framing failures.')

No framing failures.


In [22]:
# Advisory anti-hallucination check — runs across the positive-result queries
# from sections 2-4 and warns when the LLM narrative cites an address that
# isn't in the row table. Emits warnings in a DataFrame; does NOT fail the
# notebook because narrative-address parsing is loose by design.

ANTI_HALLUC_CASES = [
    'Find a 4-room flat in Bishan under $900k near MRT',
    'Show me properties near POI CHING SCHOOL',
    'Show me flats near NANYANG PRIMARY SCHOOL',
    'Find a 3 room flat in Hougang under $500k',
    'Show me a 4 room flat in Bishan with at least 90 sqm, within 500m of MRT, under $800k',
]
halluc_warnings = []
for q in ANTI_HALLUC_CASES:
    try:
        res = post_chat(q, jwt=JWT_BHUVESH)
    except Exception as e:
        halluc_warnings.append({'q': q[:55], 'rows': [], 'narrative': [], 'orphans': [], 'error': str(e)})
        continue
    md = res['markdown']
    rows = set(a.upper() for a in addresses_from_rows(md))
    narr = [a.upper() for a in addresses_from_narrative(md)]
    orphans = [a for a in narr if a not in rows]
    halluc_warnings.append({
        'q': q[:55] + ('...' if len(q) > 55 else ''),
        'row_count': len(rows),
        'narrative_addrs': len(narr),
        'orphan_addrs': len(orphans),
        'orphan_examples': '; '.join(orphans[:3]) if orphans else '',
    })
halluc = pd.DataFrame(halluc_warnings)
display(halluc)
if 'orphan_addrs' in halluc.columns and halluc['orphan_addrs'].sum() > 0:
    print(f"⚠ Advisory: {int(halluc['orphan_addrs'].sum())} narrative-address mentions weren't in the row table.")
    print('  This is loose parsing — investigate orphans manually.')
else:
    print('✓ No anti-hallucination warnings.')

,q,row_count,narrative_addrs,orphan_addrs,orphan_examples
0,Find a 4-room flat in Bishan under $900k near MRT,5,4,4,000 SOLD IN 2024 AT 106 BISHAN ST 12; 000 SOLD...
1,Show me properties near POI CHING SCHOOL,5,5,0,
2,Show me flats near NANYANG PRIMARY SCHOOL,5,1,0,
3,Find a 3 room flat in Hougang under $500k,5,4,1,000 AND 1 HOUGANG AVE 3
4,Show me a 4 room flat in Bishan with at least ...,3,2,1,000 IN 2024 AND 449 SIN MING AVE


⚠ Advisory: 6 narrative-address mentions weren't in the row table.
  This is loose parsing — investigate orphans manually.


## 5. Per-user authorization

JWT scoping → different shortlist responses for different users. Same query, different JWT, different `### From your shortlist near` block.

In [23]:
PERUSER_CASES = [
    {
        'name': 'poi_ching__bhuvesh_should_match',
        'q': 'Show me properties near POI CHING SCHOOL',
        'jwt': JWT_BHUVESH,
        'expect_xref': True,
        'expected_save_substring': '864 TAMPINES ST 83',
    },
    {
        'name': 'poi_ching__user_should_NOT_match',
        'q': 'Show me properties near POI CHING SCHOOL',
        'jwt': JWT_USER,
        'expect_xref': False,
    },
    {
        'name': 'maha_bodhi__user_should_match',
        'q': 'Show me properties near MAHA BODHI SCHOOL',
        'jwt': JWT_USER,
        'expect_xref': True,
        'expected_save_substring': 'UBI AVE 1',
    },
    {
        'name': 'maha_bodhi__bhuvesh_should_NOT_match',
        'q': 'Show me properties near MAHA BODHI SCHOOL',
        'jwt': JWT_BHUVESH,
        'expect_xref': False,
    },
]


def evaluate_peruser(case: dict) -> dict:
    try:
        res = post_chat(case['q'], jwt=case['jwt'])
    except Exception as e:
        return {'name': case['name'], 'pass': False, 'error': f'{type(e).__name__}: {e}', 'latency_ms': None}
    md = res['markdown']
    has_xref = '### From your shortlist near' in md
    has_save = (case['expected_save_substring'] in md) if case.get('expected_save_substring') else True
    has_historical = '### Historical comparables near' in md
    if case['expect_xref']:
        ok = has_xref and has_save and has_historical
    else:
        ok = (not has_xref) and has_historical
    return {
        'name': case['name'],
        'has_xref': has_xref,
        'has_save_substring': has_save,
        'has_historical': has_historical,
        'expected_xref': case['expect_xref'],
        'pass': ok,
        'latency_ms': res['latency_ms'],
        'markdown': res['markdown'][:600] + ('...' if len(res['markdown']) > 600 else ''),
    }


peruser = pd.DataFrame(evaluate_peruser(c) for c in PERUSER_CASES)
display(peruser)
print(f'Per-user authorization: {int(peruser["pass"].sum())}/{len(peruser)}')

,name,has_xref,has_save_substring,has_historical,expected_xref,pass,latency_ms,markdown
0,poi_ching__bhuvesh_should_match,True,True,True,True,True,5575,### From your shortlist near POI CHING SCHOOL\...
1,poi_ching__user_should_NOT_match,False,True,True,False,True,5220,### Historical comparables near POI CHING SCHO...
2,maha_bodhi__user_should_match,True,True,True,True,True,5671,### From your shortlist near MAHA BODHI SCHOOL...
3,maha_bodhi__bhuvesh_should_NOT_match,False,True,True,False,True,5438,### Historical comparables near MAHA BODHI SCH...


Per-user authorization: 4/4


In [24]:
# Show markdown for any peruser failure.
_failures = peruser.loc[~peruser['pass']]
if len(_failures):
    print(f'{len(_failures)} peruser failure(s):\n')
    for _, row in _failures.iterrows():
        print('─' * 80)
        print(f"name       : {row.get('name', '')}")
        print(f"has_xref   : {row.get('has_xref', '')}")
        print(f"has_save_substring : {row.get('has_save_substring', '')}")
        print(f"expected_xref : {row.get('expected_xref', '')}")
        print(f"latency_ms : {row.get('latency_ms', '')}")
        print(f"markdown   :\n{row.get('markdown', '(no markdown captured)')}")
else:
    print('No peruser failures.')

No peruser failures.


## 6. Latency profile

Bucket every call from sections 2-5 by surface and report median / p95 / max. Budgets are flagged advisory — exceeding them is diagnostic, not a hard fail.

In [25]:
BUDGETS = {
    'smalltalk':     {'median_ms': 100,  'p95_ms': 500},
    'shortlist':     {'median_ms': 1500, 'p95_ms': 3000},
    'tool':          {'median_ms': 1500, 'p95_ms': 3000},
    'search':        {'median_ms': 7000, 'p95_ms': 9000},
    'search_school': {'median_ms': 7000, 'p95_ms': 9000},
}


def bucket(intent: str) -> str:
    if intent == 'smalltalk':
        return 'smalltalk'
    if intent in ('shortlist',):
        return 'shortlist'
    if intent in ('predict', 'cbr', 'shap'):
        return 'tool'
    if intent == 'search_school':
        return 'search_school'
    return 'search'


lat_rows: list[dict] = []
for _, r in stage1.iterrows():
    if pd.notna(r.get('latency_ms')):
        lat_rows.append({'surface': bucket(r['actual_intent']), 'latency_ms': int(r['latency_ms'])})
for _, r in tools.iterrows():
    if pd.notna(r.get('latency_ms')):
        # Tool name is already explicit
        srf = 'tool' if r['name'] in ('predict', 'cbr', 'shap', 'predict_from_shortlist') else (
              'shortlist' if 'shortlist' in r['name'] or 'town' in r['name'] else 'search_school')
        lat_rows.append({'surface': srf, 'latency_ms': int(r['latency_ms'])})
for _, r in framing.iterrows():
    if pd.notna(r.get('latency_ms')):
        lat_rows.append({'surface': 'search', 'latency_ms': int(r['latency_ms'])})
for _, r in peruser.iterrows():
    if pd.notna(r.get('latency_ms')):
        lat_rows.append({'surface': 'search_school', 'latency_ms': int(r['latency_ms'])})

lat_df = pd.DataFrame(lat_rows)
if len(lat_df):
    profile = (
        lat_df.groupby('surface')['latency_ms']
        .agg(n='count', median='median', p95=lambda s: int(s.quantile(0.95)), max='max')
        .reset_index()
    )
    profile['median_budget'] = profile['surface'].map(lambda s: BUDGETS.get(s, {}).get('median_ms'))
    profile['p95_budget'] = profile['surface'].map(lambda s: BUDGETS.get(s, {}).get('p95_ms'))
    profile['median_in_budget'] = profile['median'] <= profile['median_budget']
    profile['p95_in_budget'] = profile['p95'] <= profile['p95_budget']
    display(profile)
    out_of_budget = profile.loc[~(profile['median_in_budget'] & profile['p95_in_budget'])]
    if len(out_of_budget):
        print(f"⚠ {len(out_of_budget)} surface(s) exceeded budget — diagnostic, not a hard fail.")
    else:
        print('✓ All surfaces within latency budget.')
else:
    print('No latency data collected.')

,surface,n,median,p95,max,median_budget,p95_budget,median_in_budget,p95_in_budget
0,search,11,4323.0,5252,5444,7000,9000,True,True
1,search_school,9,5220.0,5652,5671,7000,9000,True,True
2,shortlist,5,88.0,114,115,1500,3000,True,True
3,smalltalk,2,4.5,5,6,100,500,True,True
4,tool,7,92.0,137,150,1500,3000,True,True


✓ All surfaces within latency budget.


## 7. Combined summary

Single headline plus per-section breakdowns.

In [26]:
stage1_intent = f"{int(stage1['intent_match'].sum())}/{len(stage1)}"
stage1_filter = f"{int(stage1['filter_match'].sum())}/{len(stage1)}"
tool_pass     = f"{int(tools['pass'].sum())}/{len(tools)}"
framing_pass  = f"{int(framing['pass'].sum())}/{len(framing)}"
halluc_orphan_total = int(halluc['orphan_addrs'].sum()) if 'orphan_addrs' in halluc.columns else 0
peruser_pass  = f"{int(peruser['pass'].sum())}/{len(peruser)}"

median_overall = int(lat_df['latency_ms'].median()) if len(lat_df) else None
p95_overall    = int(lat_df['latency_ms'].quantile(0.95)) if len(lat_df) else None

headline = pd.DataFrame([{
    'stage1_intent_acc':   stage1_intent,
    'stage1_filter_acc':   stage1_filter,
    'tool_dispatch':       tool_pass,
    'framing_pass':        framing_pass,
    'anti_halluc_orphans': halluc_orphan_total,
    'per_user_pass':       peruser_pass,
    'median_latency_ms':   median_overall,
    'p95_latency_ms':      p95_overall,
}])
display(headline)

# Quick green/red summary
criteria = [
    ('stage1_intent ≥ 18/20',     int(stage1['intent_match'].sum()) >= 18),
    ('stage1_filter ≥ 18/20',     int(stage1['filter_match'].sum()) >= 18),
    ('tool_dispatch = 7/7',       int(tools['pass'].sum()) == len(tools)),
    ('framing = 3/3',             int(framing['pass'].sum()) == len(framing)),
    ('per_user = 4/4',            int(peruser['pass'].sum()) == len(peruser)),
]
for label, ok in criteria:
    print(f"  {'✓' if ok else '✗'}  {label}")
all_green = all(ok for _, ok in criteria)
print()
print('🟢 Demo-ready' if all_green else '🟠 Review the failing rows above before recording.')

,stage1_intent_acc,stage1_filter_acc,tool_dispatch,framing_pass,anti_halluc_orphans,per_user_pass,median_latency_ms,p95_latency_ms
0,20/20,19/20,7/7,3/3,6,4/4,2763,5592


  ✓  stage1_intent ≥ 18/20
  ✓  stage1_filter ≥ 18/20
  ✓  tool_dispatch = 7/7
  ✓  framing = 3/3
  ✓  per_user = 4/4

🟢 Demo-ready
